# Notebook 02: Training and Experimentation

**Paper 3** — Physics-Informed Neural Networks for Cloud Droplet Profile Retrieval  
Andrew J. Buggee, LASP / CU Boulder

This notebook:
1. Loads training data and creates DataLoaders
2. Defines and trains the retrieval network interactively
3. Visualizes training curves and loss components
4. Evaluates on test data and inspects predicted profiles
5. Loads checkpoints from Alpine SLURM runs for analysis

For long training runs, use `train.py` with SLURM on Alpine.
Use this notebook for quick iteration and hyperparameter exploration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from pathlib import Path
import sys
import time

sys.path.insert(0, str(Path('.').resolve().parent))
from src.models import (
    DropletProfileNetwork, CombinedLoss, RetrievalConfig,
    PhysicsLoss, SupervisedLoss
)
from src.data import create_dataloaders, adiabatic_profile, DEFAULT_N_LEVELS

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data

In [ ]:
# Path to HDF5 training data (from Notebook 01)
H5_PATH = '../data/libradtran_training_modis7.h5'

train_loader, val_loader, test_loader = create_dataloaders(
    h5_path=H5_PATH,
    batch_size=256,
    train_frac=0.8,
    val_frac=0.1,
    num_workers=0,   # Use 0 for Jupyter (avoid multiprocessing issues)
)

print(f"Train batches: {len(train_loader)} ({len(train_loader.dataset)} samples)")
print(f"Val batches:   {len(val_loader)} ({len(val_loader.dataset)} samples)")
print(f"Test batches:  {len(test_loader)} ({len(test_loader.dataset)} samples)")

# Inspect one batch
x_sample, prof_sample, tau_sample = next(iter(train_loader))
print(f"\nInput shape:   {x_sample.shape}")
print(f"Profile shape: {prof_sample.shape}")
print(f"Tau shape:     {tau_sample.shape}")

## 2. Create Model and Loss Function

In [ ]:
# Model configuration
config = RetrievalConfig(
    n_wavelengths=7,
    n_geometry_inputs=4,
    n_levels=10,
    hidden_dims=(256, 256, 256, 256),
    dropout=0.1,
    activation='gelu',
)

model = DropletProfileNetwork(config).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"Input dim: {config.input_dim}")
print(f"Output: {config.n_levels} profile levels + 1 tau_c")

In [ ]:
# Quick sanity check: forward pass
with torch.no_grad():
    output = model(x_sample.to(device))

print("Output keys:", list(output.keys()))
print(f"Profile shape: {output['profile'].shape}")
print(f"Profile range: [{output['profile'].min():.2f}, {output['profile'].max():.2f}] μm")
print(f"Tau shape:     {output['tau_c'].shape}")
print(f"Tau range:     [{output['tau_c'].min():.2f}, {output['tau_c'].max():.2f}]")

In [ ]:
# Loss function — Stage 1 (no emulator)
# Experiment with these weights!
criterion = CombinedLoss(
    config=config,
    lambda_physics=0.1,
    lambda_monotonicity=1.0,
    lambda_adiabatic=0.5,
    lambda_smoothness=0.1,
    lambda_emulator_data=0.0,  # Stage 1: no emulator
).to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

## 3. Training Loop

In [ ]:
N_EPOCHS = 100  # Increase for real training
history = {'train': [], 'val': []}

best_val_loss = float('inf')

for epoch in range(N_EPOCHS):
    # --- Train ---
    model.train()
    train_loss_sum = 0
    n_train = 0
    for x, prof_true, tau_true in train_loader:
        x, prof_true, tau_true = x.to(device), prof_true.to(device), tau_true.to(device)
        
        optimizer.zero_grad()
        output = model(x)
        losses = criterion(output, prof_true, tau_true)
        losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss_sum += losses['total'].item()
        n_train += 1
    
    # --- Validate ---
    model.eval()
    val_loss_sum = 0
    val_losses_detail = {}
    n_val = 0
    with torch.no_grad():
        for x, prof_true, tau_true in val_loader:
            x, prof_true, tau_true = x.to(device), prof_true.to(device), tau_true.to(device)
            output = model(x)
            losses = criterion(output, prof_true, tau_true)
            val_loss_sum += losses['total'].item()
            for k, v in losses.items():
                val_losses_detail[k] = val_losses_detail.get(k, 0) + v.item()
            n_val += 1
    
    train_loss = train_loss_sum / n_train
    val_loss = val_loss_sum / n_val
    val_detail = {k: v/n_val for k, v in val_losses_detail.items()}
    
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../checkpoints/best_model_notebook.pt')
    
    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
              f"Phys: {val_detail.get('physics_total', 0):.6f} | LR: {lr:.1e}")

print(f"\nBest validation loss: {best_val_loss:.6f}")

## 4. Visualize Training Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.semilogy(history['train'], label='Train', linewidth=1.5)
ax.semilogy(history['val'], label='Validation', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Total Loss', fontsize=12)
ax.set_title('Training Progress', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('../checkpoints/best_model_notebook.pt',
                                  map_location=device, weights_only=True))
model.eval()

# Collect predictions on test set
all_profiles_pred = []
all_profiles_true = []
all_tau_pred = []
all_tau_true = []

with torch.no_grad():
    for x, prof_true, tau_true in test_loader:
        output = model(x.to(device))
        # Denormalize profiles
        prof_pred = output['profile'].cpu().numpy()
        prof_true_phys = prof_true.numpy() * (config.re_max - config.re_min) + config.re_min
        
        tau_pred = output['tau_c'].squeeze(-1).cpu().numpy()
        tau_true_phys = tau_true.numpy() * (config.tau_max - config.tau_min) + config.tau_min
        
        all_profiles_pred.append(prof_pred)
        all_profiles_true.append(prof_true_phys)
        all_tau_pred.append(tau_pred)
        all_tau_true.append(tau_true_phys)

profiles_pred = np.concatenate(all_profiles_pred)
profiles_true = np.concatenate(all_profiles_true)
tau_pred = np.concatenate(all_tau_pred)
tau_true = np.concatenate(all_tau_true)

print(f"Test samples: {len(profiles_pred)}")
print(f"Profile RMSE: {np.sqrt(np.mean((profiles_pred - profiles_true)**2)):.3f} μm")
print(f"Tau RMSE:     {np.sqrt(np.mean((tau_pred - tau_true)**2)):.3f}")

In [ ]:
# Compare predicted vs true profiles for a few test cases
n_show = 6
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
z_norm = np.linspace(0, 1, config.n_levels)
rng_plot = np.random.default_rng(0)
idx = rng_plot.choice(len(profiles_pred), size=n_show, replace=False)

for k, (ax, i) in enumerate(zip(axes.flat, idx)):
    ax.plot(profiles_true[i], z_norm, 'ko-', markersize=4, linewidth=1.5, label='True')
    ax.plot(profiles_pred[i], z_norm, 's--', color='#10B981', markersize=4,
            linewidth=1.5, label='PINN')
    
    # Overlay adiabatic reference using true r_top and r_bot
    r_adiab = adiabatic_profile(profiles_true[i, 0], profiles_true[i, -1], config.n_levels)
    ax.plot(r_adiab, z_norm, ':', color='gray', linewidth=1, alpha=0.6, label='Adiabatic')
    
    ax.set_xlabel('re (μm)', fontsize=10)
    if k % 3 == 0:
        ax.set_ylabel('Depth (0=top, 1=base)', fontsize=10)
    ax.invert_yaxis()
    ax.set_title(f'τ_true={tau_true[i]:.1f}, τ_pred={tau_pred[i]:.1f}', fontsize=10)
    if k == 0:
        ax.legend(fontsize=9)

plt.suptitle('Predicted vs True Droplet Profiles (Test Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Load Checkpoint from Alpine

After running `train.py` on Alpine via SLURM, download the checkpoint
and load it here for analysis.

In [ ]:
# Example: load a checkpoint from an Alpine training run
# CHECKPOINT_PATH = '../checkpoints/20260315_143022/best_model.pt'
#
# ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
# 
# # Recreate model with saved config
# saved_config = RetrievalConfig(**ckpt['model_config'])
# model_alpine = DropletProfileNetwork(saved_config).to(device)
# model_alpine.load_state_dict(ckpt['model_state_dict'])
# model_alpine.eval()
# 
# print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
# print(f"Validation loss: {ckpt['val_loss']:.6f}")